# step 2 — RQ2 모델 다양성: stable-code-instruct-3b  (옵션 B: 마지막 토큰 치환)

**대응 RQ:** RQ2 — step1 결론(단일 층 · Value 경로 · 형태 · 방향)이 **Qwen 특성인지 일반 원리인지.**

**이 노트북:** `stabilityai/stable-code-instruct-3b`에서 step1을 반복. **데이터셋·조건은 step1과 동일**(POOL n=0, pos-camel-weak, seed 0–9), 모델만 변수.

> **왜 옵션 B(token_unit='last')인가:** stable/granite/deepseek 토크나이저는 camel/snake 이름을 **다른 토큰 수**로 쪼개(밑줄 분리) all 모드 정렬이 **전부 스킵**됐다(1차 실행 결과 무효). 옵션 B는 이름의 **마지막 토큰 하나만** 치환 → 토큰 수가 달라도 항상 1:1 정렬. 표기 차이(`matrix`↔`Matrix`)가 주로 마지막 토큰에 담긴다.
> **주의:** 이전 all-token 결과(`results/step2_stable/`에서 `tok-last`가 **없는** 파일)는 정렬 0으로 무효 → 지워도 된다. 새 결과는 슬러그에 `tok-last`가 붙어 구분된다.
> **sanity(셀 4)가 강화됨:** `S_깨끗>S_위반` **뿐 아니라 `n_substituted_tokens>0`(정렬 성공)까지** 봐야 PASS.
> **정밀도:** StableLM 계열 fp16 NaN 시 dtype='bfloat16' 재시도.


In [ ]:
# 환경 설정
!pip install -q transformers accelerate torch matplotlib pandas
import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
SEED=0; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); print('seed', SEED)

In [ ]:
# 저장소 클론 및 브랜치 체크아웃 (step2/stable-code-3b)
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
!git fetch --quiet origin step2/stable-code-3b
!git checkout step2/stable-code-3b
!git pull --quiet origin step2/stable-code-3b
!pip install -e . -q
import sys; sys.path.insert(0, 'src')
!git log --oneline -1

In [ ]:
# 조건 설정 — step1과 동일 + token_unit='last'(옵션 B). 결과는 STEP 폴더에.
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation,
                                Intervention, InterventionKind)

MODEL = ModelSpec(name='stabilityai/stable-code-instruct-3b', family='stability', dtype='float16')
DONORS = ['compliant', 'unrelated_camel']
SEEDS = list(range(10))
STEP = 'step2_stable'          # 결과 저장 폴더 results/step2_stable/
TOKEN_UNIT = 'last'           # 옵션 B: 마지막 토큰만 치환(토크나이저 무관하게 1:1 정렬)

def _pre():  return PrecedingCode(n_compliant=0, n_functions=12, composition=Composition.POOL)
def _ins():  return Instruction(form=InstructionForm.POSITIVE, target_notation=Notation.CAMEL)

def sweep_cond(donor, s):
    return Condition(model=MODEL, preceding=_pre(), instruction=_ins(),
                     intervention=Intervention(kind=InterventionKind.KEY_VALUE, layers='sweep', donor=donor),
                     seed=s, token_unit=TOKEN_UNIT)
def cosine_cond(s):
    return Condition(model=MODEL, preceding=_pre(), instruction=_ins(), seed=s, tag='vcosine', token_unit=TOKEN_UNIT)

sweep_conditions  = [sweep_cond(d, s) for d in DONORS for s in SEEDS]
cosine_conditions = [cosine_cond(s) for s in SEEDS]

PREDICTION = ('옵션 B(마지막 토큰). step1(Qwen)과 동일 패턴 예상: 회복률 단일/소수 층 국소화, '
              'Value 우세, graft 무관(형태), 피크 층 코사인 국소 딥. 비교는 상대 위치로.')
print(len(sweep_conditions), '스윕 +', len(cosine_conditions), '코사인  | token_unit=', TOKEN_UNIT, '| STEP=', STEP)

In [ ]:
# 셀 4 — SANITY (강화). 정렬 성공(n_sub>0) AND S_깨끗>S_위반 이어야 PASS.
from harness import run, ResultRecord, save_result, result_path
from harness.model import load_model

handle = load_model(MODEL)
gqa = handle.gqa_info()
print('=== (a) config.json GQA ===')
print(f'  layers={handle.num_layers}  attn={gqa.num_attention_heads}  kv={gqa.num_key_value_heads}'
      f'  head_dim={gqa.head_dim}  group={gqa.group_size}  is_gqa={gqa.is_gqa}')
print('=== (b) chat template ===', 'OK' if getattr(handle.tokenizer,'chat_template',None) else '없음(!)')

print('=== (c) 정렬률 + S_깨끗>S_위반 (seed0, compliant, token_unit=last) ===')
c0 = sweep_cond('compliant', 0)
out0 = run(c0, handle=handle)
save_result(ResultRecord(condition=out0.condition, metrics=out0.metrics, step=STEP, rq='RQ2', prediction=PREDICTION))
ex = out0.metrics.extra
n_names=len(ex['viol_names'])
print(f'  이름 {n_names}개 -> 정렬 토큰 {ex["n_substituted_tokens"]}개, 스킵 {len(ex["skipped_names"])}개')
print(f'  S_깨끗 = {ex["S_clean"]:+.3f}  |  S_위반 = {ex["S_base"]:+.3f}')
aligned = ex['n_substituted_tokens'] > 0
sflip = (ex['S_clean'] is not None and ex['S_base'] is not None
         and ex['S_clean']==ex['S_clean'] and ex['S_clean']>ex['S_base'])
print(f'  정렬 성공? {aligned}  |  선호 뒤집힘? {sflip}')
print('\n>>> SANITY', 'PASS -> 셀 5 진행' if (aligned and sflip)
      else 'FAIL -> 정렬 0이면 last 모드도 실패(이름 못찾음)/S NaN. 원인 확인 후 진행(강행 금지)')

In [ ]:
# 셀 5 — 전체 실행 (sanity PASS 후). 재개 가능.
new=skipped=0
for i,c in enumerate(sweep_conditions,1):
    if result_path(c, step=STEP).exists(): skipped+=1
    else:
        out=run(c, handle=handle)
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics, step=STEP, rq='RQ2', prediction=PREDICTION)); new+=1
    if i%4==0 or i==len(sweep_conditions): print(f'[스윕 {i}/{len(sweep_conditions)}] 새 {new} / 건너뜀 {skipped}')
cnew=cskip=0
for i,c in enumerate(cosine_conditions,1):
    if result_path(c, step=STEP).exists(): cskip+=1
    else:
        out=run(c, handle=handle, mode='vcosine')
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics, step=STEP, rq='RQ2', prediction=PREDICTION)); cnew+=1
print(f'코사인: 새 {cnew} / 건너뜀 {cskip}'); print('완료.')

In [ ]:
# 셀 6 — 요약(피크 상대 위치 · K/V · 코사인). token_unit=last 결과만 로드.
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import defaultdict
from harness import result_path
from harness.results import load_result
from harness.intervention import peak_layer

sweep_recs=[load_result(result_path(c, step=STEP)) for c in sweep_conditions if result_path(c, step=STEP).exists()]
cosine_recs=[load_result(result_path(c, step=STEP)) for c in cosine_conditions if result_path(c, step=STEP).exists()]
NL=handle.num_layers; KINDS=['key','value','key_value']; COL={'key':'#2563C9','value':'#C6771A','key_value':'#2E7D52'}
dpres=sorted({r.condition.intervention.donor for r in sweep_recs})
print(f'로드 스윕 {len(sweep_recs)} / 코사인 {len(cosine_recs)}  NL={NL}  정렬토큰(seed0)={sweep_recs[0].metrics.extra["n_substituted_tokens"]}')

agg={d:{k:defaultdict(list) for k in KINDS} for d in dpres}
for r in sweep_recs:
    d=r.condition.intervention.donor
    for L,flat in r.metrics.per_layer.items():
        for k in KINDS:
            if f'{k}__recovery' in flat: agg[d][k][int(L)].append(flat[f'{k}__recovery'])
def curve(d,k):
    Ls=sorted(agg[d][k]); return Ls,[float(np.mean(agg[d][k][L])) for L in Ls]

print('=== 피크 층 (rel=peak/(NL-1)) ===')
rows=[]
for d in dpres:
    for k in KINDS:
        Ls,vals=curve(d,k); pk=peak_layer(dict(zip(Ls,vals)))
        rows.append({'donor':d,'kind':k,'peak_L':pk[0],'rel':round(pk[0]/(NL-1),3),'peak_rec':round(pk[1],3)})
print(pd.DataFrame(rows).to_string(index=False))

cos=defaultdict(list)
for r in cosine_recs:
    for L,flat in r.metrics.per_layer.items():
        if flat.get('v_cosine') is not None: cos[int(L)].append(flat['v_cosine'])
cLs=sorted(cos); cvals=[float(np.mean(cos[L])) for L in cLs]

npan=len(dpres)+1
fig,ax=plt.subplots(1,npan,figsize=(5*npan,4))
for j,d in enumerate(dpres):
    for k in KINDS:
        Ls,vals=curve(d,k); ax[j].plot(Ls,vals,color=COL[k],label=k,marker='.',ms=4)
    ax[j].axhline(0,color='#aaa',lw=.6); ax[j].axhline(1,color='#aaa',ls='--',lw=.6)
    ax[j].set_title(f'stable-3b (last-tok) — {d}'); ax[j].set_xlabel(f'layer/{NL}'); ax[j].legend(fontsize=8)
if cLs:
    ax[-1].plot(cLs,cvals,color='#7B3FA0',marker='.',ms=4)
ax[-1].set_title('v cosine (last-tok)'); ax[-1].set_xlabel(f'layer/{NL}'); ax[-1].set_ylim(0,1.05)
plt.tight_layout(); plt.savefig('step2_stable_lasttok_summary.png',dpi=110); plt.show()

sc=float(np.mean([r.metrics.extra['S_clean'] for r in sweep_recs])); sb=float(np.mean([r.metrics.extra['S_base'] for r in sweep_recs]))
print(f'sanity S_clean {sc:+.2f} > S_base {sb:+.2f} : {sc>sb}')

In [ ]:
# 결과 다운로드
import shutil
shutil.make_archive('step2_stable_results', 'zip', 'results/step2_stable')
try:
    from google.colab import files; files.download('step2_stable_results.zip')
except Exception as e:
    print('Colab 아님:', e)